# FuzzyDiff on SD3-Medium

Settings: GPU T4 x2, Internet on. Add a Kaggle Secret named `HF_TOKEN`, from a Hugging Face account that has accepted the SD3-Medium licence.

All logic lives in `code/`. Results from this notebook use provisional calibration: they show the pipeline runs and are never paper evidence.


In [ ]:
BRANCH = 'fuzzysd3'
OUT = '/kaggle/working/outputs'
RESUME = ''  # path to an earlier session's outputs, attached as a Kaggle Dataset
VAE_DEVICE = 'cuda:1'  # the transformer stays on cuda:0

!git clone -q --branch {BRANCH} https://github.com/sunzidulislam/fuzzydiff.git /kaggle/working/fuzzydiff
%cd /kaggle/working/fuzzydiff
!pip install -q -r requirements-kaggle.txt
# CLIP-IQA only; --no-deps because piq would pull in a torchvision that does not match this torch.
!pip install -q --no-deps piq==0.8.0
# Kaggle's torchvision is built for its own torch and crashes on import with this one; transformers
# imports it whenever it is present, and the pipeline does not need it.
!pip uninstall -y -q torchvision
!python -I -c "from transformers import CLIPSegForImageSegmentation, SiglipModel; from diffusers import StableDiffusion3Pipeline; print('libraries ok')"
!git log -1 --oneline


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi -L


## 1. Verify before anything else

`test_processor.py` runs on CPU. `verify_checkpoint.py` checks the real weights and reports memory: look at `mem_guided_gb` before going on. Its output goes to `/kaggle/working/verify`.


In [ ]:
!python -I code/test_processor.py | tail -5
!mkdir -p /kaggle/working/verify
!python -I code/verify_checkpoint.py --model stabilityai/stable-diffusion-3-medium-diffusers --vae-device {VAE_DEVICE} --out /kaggle/working/verify | tail -30


## 1b. Check phrase tokens on the real T5 tokenizer

CPU tests use fake offsets, so this confirms each phrase maps to its own tokens. Each line should show the phrase it names.


In [ ]:
import sys
sys.path.insert(0, 'code')
from transformers import AutoTokenizer
from grounding import map_tokens, phrase_spans
from predicates import translate
tok = AutoTokenizer.from_pretrained('stabilityai/stable-diffusion-3-medium-diffusers', subfolder='tokenizer_3')
for prompt in ['a red mug and a red teapot', 'a greenish-gray mug and a turquoise-green teapot', 'a kitten slightly larger than a pillow']:
    enc = tok(prompt, padding='max_length', max_length=256, truncation=True, return_offsets_mapping=True)
    tokens, bad = map_tokens([tuple(o) for o in enc['offset_mapping']], phrase_spans(prompt, translate(prompt)))
    print(prompt, 'unmapped:', bad)
    for phrase, idx in tokens.items():
        print('  ', phrase, '->', tok.decode([enc['input_ids'][j - 77] for j in idx]))


In [ ]:
# Stops here if the check above failed: the rest of the notebook means nothing without it.
import json
check = json.load(open('/kaggle/working/verify/checkpoint_check.json'))
print({k: check.get(k) for k in ('mem_unguided_gb', 'mem_guided_gb', 'mem_guided_vae_gpu_gb', 'mem_sar_gb', 'identity_out', 'identity_grad', 'identity_grad_noise_floor', 'readout_grad', 'fd_grad', 'image_identity', 'token_index')})


## 1c. One two-object image through every stage

A fast check of memory and of the CLIPSeg, SigLIP, and SAR path on real weights, before any long run. `peak_by_gpu` in the printed output shows both GPUs.


In [ ]:
!python -I code/run_experiment.py --suite plain_color --group dev --n 1 --arms stage1,stage1_sar --out {OUT}/smoke --vae-device {VAE_DEVICE}
import glob
for p in glob.glob(f'{OUT}/smoke/*/*/meta.json'):
    m = json.load(open(p))
    print(m['arm'], '| failed:', m.get('failed'), '| time_s:', m.get('time_s'), '| peak_by_gpu:', m.get('peak_by_gpu'),
          '| bias_steps:', m.get('bias_steps'), '| no_op:', m.get('no_op_reason'), '| fallback:', m.get('used_fallback'))
    if m.get('traceback'):
        print(m['traceback'][-1500:])


## 1d. Measure presence and attention thresholds

Generates 20 unguided development images, then scores objects that are in each image against objects that are not, and checks where attention lands. Look at `{OUT}/probe/images`: the labels assume each image shows the objects its prompt names.


In [ ]:
!python -I code/threshold_probe.py --out {OUT}/probe --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn'


## 2. M1 tuning

Three step sizes, two points to start guidance (sigma 0.75 and 0.5), and two directions, on 5 separate prompts. Compare the images as well as the numbers: lightness can reach its target while the scene turns into something else.


In [ ]:
resume = f'--resume-from {RESUME}/m1_tune' if RESUME else ''
!python -I code/run_experiment.py --suite m1_tune --out {OUT}/m1_tune --vae-device {VAE_DEVICE} {resume}


Set the chosen `cfgEtaZero`, `cfgSigmaHi` and `cfgMaxPert` in `configs/provisional.yaml` before the next cell.


In [ ]:
import json, glob, collections
rows = collections.defaultdict(list)
for p in glob.glob(f'{OUT}/m1_tune/*@*/*/meta.json'):
    m = json.load(open(p))
    if 'dy' in m:
        rows[m['arm']].append((m['ds'], m['dy'], m['time_s']))
for arm, r in sorted(rows.items()):
    print(arm, 'n', len(r), 'mean dy/ds', round(sum(b / a for a, b, _ in r) / len(r), 3), 'time_s', round(sum(t for *_, t in r) / len(r), 1))


## 3. M1 evaluation and ladder


In [ ]:
resume = f'--resume-from {RESUME}/m1' if RESUME else ''
!python -I code/run_experiment.py --suite m1_eval --out {OUT}/m1 --vae-device {VAE_DEVICE} {resume}


## 4. Blind rating page

Download `rate.html`, rate on your own machine, and save the exported text as `ratings.json`. Don't open `rate.key.json` until you've finished rating.


In [ ]:
import sys, glob
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_rating_page
images = [Path(p) for p in glob.glob(f'{OUT}/m1/*/*/image.png') if '/ladder' not in p]
build_rating_page(images, Path(f'{OUT}/m1/rate.html'), seed=0)
print(len(images), 'images')


## 5. Colour, size, and SAR on development prompts

Debug on plain colours, then compare on mixtures. All arms use the same prompts and seeds. The last three arms refine the same Stage 1 image from the same re-noised latent.


In [ ]:
resume = f'--resume-from {RESUME}/plain' if RESUME else ''
!python -I code/run_experiment.py --suite plain_color --group dev --n 5 --out {OUT}/plain --vae-device {VAE_DEVICE} {resume}


In [ ]:
resume = f'--resume-from {RESUME}/mixture' if RESUME else ''
!python -I code/run_experiment.py --suite mixture --group dev --n 20 --out {OUT}/mixture --vae-device {VAE_DEVICE} {resume}


In [ ]:
resume = f'--resume-from {RESUME}/size' if RESUME else ''
!python -I code/run_experiment.py --suite size --group dev --n 10 --arms unguided,stage1 --out {OUT}/size --vae-device {VAE_DEVICE} {resume}


## 6. Blind SAR comparison

Download `compare.html` and rate it on your own machine. Don't open `compare.key.json` until you've finished. Check `fallback_rate` and `no_bias_share` too: a SAR run that always falls back or never applies its bias looks harmless while doing nothing.


In [ ]:
import importlib, json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_comparison_page
from run_experiment import summarize_arms
build_comparison_page(f'{OUT}/mixture', Path(f'{OUT}/mixture/compare.html'), seed=0)
print(json.dumps(summarize_arms(f'{OUT}/mixture'), indent=1))


## 7. Your own prompt

Runs one prompt through the whole pipeline and shows SD3 alone, Stage 1, and Stage 1 + SAR side by side. Results use provisional calibration unless `--calibration` is given, so they are for looking at, not for the paper.


In [ ]:
PROMPT = 'a slightly red mug and a blue teapot'
SEED = 0
!python -I code/run_experiment.py --prompt "{PROMPT}" --seeds {SEED} --out {OUT}/mine --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn'


In [ ]:
import hashlib, json
import matplotlib.pyplot as plt
from PIL import Image
image_id = f"{hashlib.sha256(PROMPT.encode()).hexdigest()[:12]}_s{SEED}"
arms = [('unguided', 'SD3 alone'), ('stage1', 'Stage 1'), ('stage1_sar', 'Stage 1 + SAR')]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (arm, title) in zip(axes, arms):
    d = f'{OUT}/mine/{arm}/{image_id}'
    m = json.load(open(f'{d}/meta.json'))
    if 'failed' in m:
        ax.set_title(f"{title}: failed ({m['failed']})")
    else:
        ax.imshow(Image.open(f'{d}/image.png'))
        note = ' (fell back to Stage 1)' if m.get('used_fallback') else ''
        ax.set_title(title + note)
    ax.axis('off')
plt.suptitle(PROMPT)
plt.show()


## 8. FMAS pilot

Ten soft-attribute development prompts through SD3, Stage 1, Stage 1 + plain pass, and Stage 1 + SAR. Download `fmas.html` and rate it on your own machine. Don't open `fmas.key.json` until you've finished. Author ratings with provisional calibration: a pilot, never paper evidence.


In [ ]:
!python -I code/run_experiment.py --suite soft --group dev --n 10 --arms unguided,stage1,stage1_plain,stage1_sar --out {OUT}/fmas --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_fmas_page
build_fmas_page(f'{OUT}/fmas', Path(f'{OUT}/fmas/fmas.html'), seed=0)
print('rating page:', f'{OUT}/fmas/fmas.html')


In [ ]:
# Press "Show ratings" on the page, then paste the text between the quotes.
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import score_fmas
Path(f'{OUT}/fmas/ratings.json').write_text(RATINGS)
print(json.dumps(score_fmas(f'{OUT}/fmas/ratings.json', f'{OUT}/fmas/fmas.key.json'), indent=1))


## 9. Success Rate pilot

Uses the images from section 8, in a new order. Download `success.html` and rate it on your own machine. Don't open `success.key.json` until you've finished.


In [ ]:
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_success_page
build_success_page(f'{OUT}/fmas', Path(f'{OUT}/fmas/success.html'), seed=1)
print('rating page:', f'{OUT}/fmas/success.html')


In [ ]:
# Press "Show ratings" on the page, then paste the text between the quotes.
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import score_success
Path(f'{OUT}/fmas/success_ratings.json').write_text(RATINGS)
print(json.dumps(score_success(f'{OUT}/fmas/success_ratings.json', f'{OUT}/fmas/success.key.json'), indent=1))


## 10. Automatic similarity and CLIP-IQA

Text-image (CLIP ViT-B/16), text-text (BLIP caption, then CLIP ViT-B/16 text), and CLIP-IQA (piq, CLIP ResNet-50) for every image of a run, with the same paired comparisons as FMAS. Writes `similarity.json` and per-image `similarity.jsonl` with each caption.


If `{OUT}/fmas` is gone, upload `fmas.html` and `fmas.key.json` (the files you downloaded for rating) as a Kaggle dataset and add it as input. The cell then rebuilds the folder from the page. Those images are 512 px JPEGs, so CLIP-IQA values differ from a full-resolution run (each `meta.json` says `restored_from`).


In [ ]:
!pip install -q --no-deps piq==0.8.0
import glob, os
if not os.path.isdir(f'{OUT}/fmas/stage1'):
    page = glob.glob('/kaggle/input/**/fmas.html', recursive=True)
    key = glob.glob('/kaggle/input/**/fmas.key.json', recursive=True)
    assert page and key, 'no fmas folder: add fmas.html and fmas.key.json as input, or rerun section 8'
    !python -I code/restore_run.py --page {page[0]} --key {key[0]} --out {OUT}/fmas
!python -I code/auto_metrics.py --run {OUT}/fmas --device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|warn'

## 11. Colour mixture pilot (Exp. ii)

Fifty colour-mixture development prompts through SD3, Stage 1, Stage 1 + plain pass, and Stage 1 + SAR (about 5 hours). One page rates missing objects, attribute leakage and Fidelity together. Download `mixture.html` and rate it on your own machine. Don't open `mixture.key.json` until you've finished. Save `{OUT}/mixture_pilot` before the session ends; rerunning the cell resumes.


In [ ]:
!python -I code/run_experiment.py --suite mixture --group dev --n 50 --arms unguided,stage1,stage1_plain,stage1_sar --out {OUT}/mixture_pilot --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_mixture_page
build_mixture_page(f'{OUT}/mixture_pilot', Path(f'{OUT}/mixture_pilot/mixture.html'), seed=3)
print('rating page:', f'{OUT}/mixture_pilot/mixture.html')


In [ ]:
# Press "Show ratings" on the page, then paste the text between the quotes.
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import score_mixture
Path(f'{OUT}/mixture_pilot/mixture_ratings.json').write_text(RATINGS)
print(json.dumps(score_mixture(f'{OUT}/mixture_pilot/mixture_ratings.json', f'{OUT}/mixture_pilot/mixture.key.json'), indent=1))


## 12. Ablations (Table 2)

Five prompts per experiment, 14 arms each: SD3, Stage 1, eight Stage-1 ablations, and four Stage-2 variants. The two validity-gate rows have no arm: provisional calibration has neither gate, so they equal Stage 1.

Each experiment takes roughly 3 hours (an estimate), so run them in separate sessions. A rerun of the same cell resumes. Save each output folder before the session ends. Don't open the `.key.json` files until you've finished rating.


In [ ]:
# 12a. Exp. (ii) colour mixtures: missing objects, leakage, Fidelity
!python -I code/run_experiment.py --suite mixture --group dev --n 5 --arms ablation --out {OUT}/ablation_mix --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import ABLATION_ARM_NAMES, build_mixture_page
build_mixture_page(f'{OUT}/ablation_mix', Path(f'{OUT}/ablation_mix/ablation_mix.html'), seed=7, arms=ABLATION_ARM_NAMES)
print('rating page:', f'{OUT}/ablation_mix/ablation_mix.html')


In [ ]:
# 12b. Exp. (iii) size relations: missing objects, size violations, Fidelity
!python -I code/run_experiment.py --suite size --group dev --n 5 --arms ablation --out {OUT}/ablation_size --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import ABLATION_ARM_NAMES, build_size_page
build_size_page(f'{OUT}/ablation_size', Path(f'{OUT}/ablation_size/ablation_size.html'), seed=8, arms=ABLATION_ARM_NAMES)
print('rating page:', f'{OUT}/ablation_size/ablation_size.html')


In [ ]:
# 12c. Press "Show ratings" on each page and paste the text. Leave a page as PASTE HERE until it is rated.
MIX_RATINGS = r'''PASTE HERE'''
SIZE_RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import ABLATION_COMPARISONS, ablation_table_rows, score_mixture
scores = {}
for name, text, field in (('ablation_mix', MIX_RATINGS, 'leakage'), ('ablation_size', SIZE_RATINGS, 'size')):
    if text.strip() == 'PASTE HERE':
        continue
    Path(f'{OUT}/{name}/{name}_ratings.json').write_text(text)
    scores[name] = score_mixture(f'{OUT}/{name}/{name}_ratings.json', f'{OUT}/{name}/{name}.key.json',
                                 field=field, comparisons=ABLATION_COMPARISONS)
    print(name, json.dumps(scores[name], indent=1))
print(ablation_table_rows(mixture=scores.get('ablation_mix'), size=scores.get('ablation_size')))


## 13. Hedge ladder (Table 3)

Six base prompts per family, each at barely < slightly < moderately < very < extremely, through SD3, SD3 + prompt weighting, Stage 1 and FuzzyDiff (Stage 1 + SAR). Prompt weighting keeps the hedge in the prompt and multiplies the embeddings of the word it modifies by 0.5, 0.75, 1.0, 1.25 or 1.5. Concept Sliders and SPO have no SD3-Medium release, so their rows stay TBD.

Each family takes roughly 3 hours (an estimate), so 13a to 13c can go in separate sessions. A rerun resumes. Save each `{OUT}/ladder_*` folder before the session ends. 13d measures every image with OWLv2 and builds one rating page from the families that are done. Don't open `ladder.key.json` until you've finished rating.


In [ ]:
# 13a. Attribute (luminance and saturation): 6 base prompts x 5 hedges x 4 methods
!python -I code/run_experiment.py --suite ladder_attribute --group dev --n 6 --arms ladder --out {OUT}/ladder_attribute --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30

In [ ]:
# 13b. Colour mixtures: 6 base prompts x 5 hedges x 4 methods
!python -I code/run_experiment.py --suite ladder_mixture --group dev --n 6 --arms ladder --out {OUT}/ladder_mixture --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30

In [ ]:
# 13c. Size relations: 6 base prompts x 5 hedges x 4 methods, on plausible object pairs
# (ladder_size_v2: the first run, in ladder_size, paired objects at random, e.g. a kitten larger than a tent)
!python -I code/run_experiment.py --suite ladder_size --group dev --n 6 --arms ladder --out {OUT}/ladder_size_v2 --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30

In [ ]:
# 13d. OWLv2 measurement (rho) and the rating page
!python -I code/ladder.py --runs {OUT}/ladder_attribute,{OUT}/ladder_mixture,{OUT}/ladder_size_v2 --out {OUT}/ladder_auto.json --device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|warn' | tail -60
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from ladder import build_ladder_page
build_ladder_page([f'{OUT}/ladder_attribute', f'{OUT}/ladder_mixture', f'{OUT}/ladder_size_v2'], Path(f'{OUT}/ladder.html'), seed=10)
print('rating page:', f'{OUT}/ladder.html')

In [ ]:
# 13e. Press "Show ratings" on the page and paste the text. Leave it as PASTE HERE to print rho alone.
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from ladder import ladder_table_rows, score_ladder
human = None
if RATINGS.strip() != 'PASTE HERE':
    Path(f'{OUT}/ladder_ratings.json').write_text(RATINGS)
    human = score_ladder(f'{OUT}/ladder_ratings.json', f'{OUT}/ladder.key.json')
    print(json.dumps(human, indent=1))
auto = json.loads(Path(f'{OUT}/ladder_auto.json').read_text())
print(ladder_table_rows(human, auto))

In [ ]:
# 13f. Size only, when attribute and colour are already rated: OWLv2 and a size-only rating page for ladder_size_v2.
# Paste its ratings below once rated; the size columns of Table 3 come from here.
SIZE_RATINGS = r'''PASTE HERE'''
!python -I code/ladder.py --runs {OUT}/ladder_size_v2 --out {OUT}/ladder_size_auto.json --device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|warn' | tail -40
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from ladder import build_ladder_page, score_ladder
if not Path(f'{OUT}/ladder_size.html').exists():
    build_ladder_page([f'{OUT}/ladder_size_v2'], Path(f'{OUT}/ladder_size.html'), seed=11)
    !cd {OUT} && zip -qr ladder_size_v2.zip ladder_size_v2 ladder_size.html ladder_size.key.json ladder_size_auto.json ladder_size_auto.jsonl
    print('rating page:', f'{OUT}/ladder_size.html')
if SIZE_RATINGS.strip() != 'PASTE HERE':
    Path(f'{OUT}/ladder_size_ratings.json').write_text(SIZE_RATINGS)
    print(json.dumps(score_ladder(f'{OUT}/ladder_size_ratings.json', f'{OUT}/ladder_size.key.json'), indent=1))

## 14. Screening and guidance sweep (before rerunning Table 2)

The Table 2 pilot could not separate the components, for two reasons. SD3 already got the colour-mixture prompts right, and Stage 1 never changed relative size: 76% of guided steps hit the step cap (median step 0.006), and guidance starts at sigma 0.75, after SD3 has laid out the scene.

- **14a** runs SD3 alone on 30 candidate prompts per experiment and keeps the 5 it gets most wrong among those where it drew every object, by FuzzyDiff's own check (every presence membership at least 0.5, lowest other membership below 0.5). About 1 hour. Report the rule with the table.
- **14b** tries four guidance settings on two size ladders (dev prompts): current, stronger steps, guidance from the first step, and both. About 3-4 hours; a rerun resumes, so it can span sessions. Save `{OUT}/sweep` before the session ends.
- **14c** scores each setting with OWLv2, not with FuzzyDiff's scorer: rho (does measured size follow the hedge) and moved (how far size changed against SD3).
- **14d** reruns Table 2 on the screened prompts with the setting checked in 15d (guidance from sigma 0.9, RMS step limit 0.02), into `ablation_mix3` and `ablation_size3`; then rate the pages and score them in 14e.

The first rerun (`ablation_mix2`, `ablation_size2`) used the provisional setting and all 14 arms gave the same picture; four of its five size prompts had an object missing. Fix the setting from 15d before generating 14d. Picking whichever setting rates best on the Table 2 prompts would tune on the test images.

In [ ]:
# 14a. Screening: SD3 alone on 30 candidates per experiment, keep the 5 it fails worst among those where it drew
# every object (a missing object gates the size or colour constraint off, so there is nothing to ablate)
!python -I code/run_experiment.py --suite mixture --group dev --n 30 --arms unguided --out {OUT}/screen_mix --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
!python -I code/run_experiment.py --suite size --group dev --n 30 --arms unguided --out {OUT}/screen_size --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
!python -I code/screen.py --run {OUT}/screen_mix --out {OUT}/screen_mix.json --keep 5 --objects-present
!python -I code/screen.py --run {OUT}/screen_size --out {OUT}/screen_size.json --keep 5 --objects-present

In [ ]:
# 14b. Sweep: SD3 and four Stage-1 settings on two size ladders (2 base prompts x 5 hedges)
import sys
sys.path.insert(0, 'code')
from sweep import SETTINGS, sweep_flags
!python -I code/run_experiment.py --suite ladder_size --group dev --n 2 --arms unguided --out {OUT}/sweep/unguided --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
for name in SETTINGS:      # to split across sessions, replace SETTINGS with e.g. ['current', 'stronger']
    flags = sweep_flags(name)
    print('=====', name, flags)
    !python -I code/run_experiment.py --suite ladder_size --group dev --n 2 --arms stage1 {flags} --out {OUT}/sweep/{name} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
!cd {OUT} && zip -qr sweep.zip sweep

In [ ]:
# 14c. Score the sweep with OWLv2. Higher rho is better (1 = size follows the hedge exactly); moved 0 means
# guidance did not change size at all. A high evidence rho with a low rho means a setting fools its own scorer.
!python -I code/sweep.py --root {OUT}/sweep --out {OUT}/sweep.json --device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|warn' | tail -12

In [ ]:
# 14d. Rerun Table 2 on the screened prompts. The setting is the one 15d checked on 10 size prompts; change it
# here only from 15d (other prompts), never from how these Table 2 images rate.
size_flags = '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04'
# an earlier start can change which objects appear, so colour keeps sigma_hi 0.75 and takes only the step limit
mix_flags = '--clip rms --dmax 0.02 --eta0 0.04'
import sys
sys.path.insert(0, 'code')
from pathlib import Path
print('size:', size_flags, '| colour:', mix_flags)
!python -I code/run_experiment.py --prompts-file {OUT}/screen_mix.json --arms ablation {mix_flags} --out {OUT}/ablation_mix3 --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
!python -I code/run_experiment.py --prompts-file {OUT}/screen_size.json --arms ablation {size_flags} --out {OUT}/ablation_size3 --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
from annotate import ABLATION_ARM_NAMES, build_mixture_page, build_size_page
build_mixture_page(f'{OUT}/ablation_mix3', Path(f'{OUT}/ablation_mix3/ablation_mix3.html'), seed=17, arms=ABLATION_ARM_NAMES)
build_size_page(f'{OUT}/ablation_size3', Path(f'{OUT}/ablation_size3/ablation_size3.html'), seed=18, arms=ABLATION_ARM_NAMES)
!cd {OUT} && zip -qr ablation3.zip ablation_mix3 ablation_size3
print('rating pages:', f'{OUT}/ablation_mix3/ablation_mix3.html', f'{OUT}/ablation_size3/ablation_size3.html')

In [ ]:
# 14e. Press "Show ratings" on each page and paste the text. Leave a page as PASTE HERE until it is rated.
MIX_RATINGS = r'''PASTE HERE'''
SIZE_RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import ABLATION_COMPARISONS, ablation_table_rows, score_mixture
scores = {}
for name, text, field in (('ablation_mix3', MIX_RATINGS, 'leakage'), ('ablation_size3', SIZE_RATINGS, 'size')):
    if text.strip() == 'PASTE HERE':
        continue
    Path(f'{OUT}/{name}/{name}_ratings.json').write_text(text)
    scores[name] = score_mixture(f'{OUT}/{name}/{name}_ratings.json', f'{OUT}/{name}/{name}.key.json',
                                 field=field, comparisons=ABLATION_COMPARISONS)
    print(name, json.dumps(scores[name], indent=1))
print(ablation_table_rows(mixture=scores.get('ablation_mix3'), size=scores.get('ablation_size3')))

## 15. Fuzzy comparison (Table 4, Exp. iii)

Ten size prompts from the development group, drawn at random (not screened, so they match how the baseline rows were chosen), each through SD3, Stage 1 (S1) and FuzzyDiff (Stage 1 + SAR) with the same checkpoint, sampler, steps, CFG, resolution and seed. The size prompts use only plausible object pairs.

This section can run in its own Kaggle session, in parallel with sections 12 to 14: it writes only to `{OUT}/exp3_v2`. Run the setup cells first. 15a takes roughly 1.5 to 2 hours; a rerun resumes. Save `exp3_v2.zip` before the session ends, and don't open `exp3_v2/size.key.json` until you've finished rating.

In [ ]:
# 15a. SD3, S1 and FuzzyDiff on 10 size prompts, then the blind rating page. The first run (exp3) used the
# provisional setting and its three rows looked the same; this one uses the setting checked in 15d.
FLAGS = '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04'
!python -I code/run_experiment.py --suite size --group dev --n 10 --arms unguided,stage1,stage1_sar --out {OUT}/exp3_v2 {FLAGS} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import EXP3_ARMS, build_size_page
build_size_page(f'{OUT}/exp3_v2', Path(f'{OUT}/exp3_v2/size.html'), seed=21, arms=EXP3_ARMS)
!cd {OUT} && zip -qr exp3_v2.zip exp3_v2 -x '*.pt'
print('rating page:', f'{OUT}/exp3_v2/size.html')

In [ ]:
# 15b. Automatic columns: text-image / text-text similarity (CLIP ViT-B/16, BLIP captions) and CLIP-IQA
!pip install -q --no-deps piq==0.8.0
!python -I code/auto_metrics.py --run {OUT}/exp3_v2 --arms unguided,stage1,stage1_sar --device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|warn' | tail -5
!cd {OUT} && zip -qr exp3_v2.zip exp3_v2 -x '*.pt'

In [ ]:
# 15c. Press "Show ratings" on the page and paste the text. Leave it as PASTE HERE to print the automatic columns alone.
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import EXP3_COMPARISONS, exp3_table_rows, score_mixture
human = None
if RATINGS.strip() != 'PASTE HERE':
    Path(f'{OUT}/exp3_v2/size_ratings.json').write_text(RATINGS)
    human = score_mixture(f'{OUT}/exp3_v2/size_ratings.json', f'{OUT}/exp3_v2/size.key.json', field='size',
                          comparisons=EXP3_COMPARISONS)
    print(json.dumps(human, indent=1))
auto_path = Path(f'{OUT}/exp3_v2/similarity.json')
auto = json.loads(auto_path.read_text()) if auto_path.exists() else None
print(exp3_table_rows(human, auto))

### 15d.0. One prompt first: does Stage 1 change the picture?

Before any 10-prompt run. One prompt, SD3 and Stage 1 under four settings, about 10 minutes per setting; the table is printed again after each one, so you can stop as soon as you have seen enough. Two runs are already known: guidance from sigma 0.75 at about 0.005 RMS per step left the picture unchanged (15a), and guidance from the first step at 0.05 RMS per step gave flat, saturated shapes. These settings lie in between.

Read the table with the sheet `exp3_one/sheet.jpg` (SD3 | each setting). `moved` is how far the latent was pushed in total (its own RMS is about 1), `score` is log(area ratio) with its target printed on top, `pixels` is the mean difference to SD3 (0-255), `sat.` the share of saturated pixels. A setting works only if the sheet shows a photo in which the size relation moved toward the prompt; a better score on an unchanged picture means the estimator moved, not the image. `exp3_one/steps.csv` has every guided step. Send me `exp3_one.zip`.

In [ ]:
# 15d.0. One prompt under four guidance settings; the table and the sheet are rebuilt after each
PROMPT = 'a kettle extremely bigger than a shoe'   # SD3 draws both objects, at nearly the same size
ONE = {'rms0.01': '--sigma-hi 1.0 --clip rms --dmax 0.01 --eta0 0.02',
       'rms0.02': '--sigma-hi 1.0 --clip rms --dmax 0.02 --eta0 0.04',
       'rms0.02_hi0.9': '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04',
       'rms0.03': '--sigma-hi 1.0 --clip rms --dmax 0.03 --eta0 0.06'}
import sys
sys.path.insert(0, 'code')
from IPython.display import Image as show, display
from layout_check import check
for k, (name, flags) in enumerate(ONE.items()):
    arms = 'unguided,stage1' if k == 0 else 'stage1'
    !python -I code/run_experiment.py --suite size --group dev --n 10 --only-prompt "{PROMPT}" --arms {arms} --out {OUT}/exp3_one/{name} {flags} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -3
    check(f'{OUT}/exp3_one')
    display(show(f'{OUT}/exp3_one/sheet.jpg'))
!cd {OUT} && zip -qr exp3_one.zip exp3_one -x '*.pt'

### 15d. Guidance that changes the layout (check before rerunning 15a)

In 15a the three methods gave nearly the same image: guidance started at sigma 0.75, after SD3 had placed the objects, and each step moved the latent by only about 0.005 RMS. On one prompt (15d.0, kettle and shoe) guidance from the very first step broke the image at every strength tried, even 0.01 RMS per step: at sigma near 1 the clean prediction is a blur, the objects are not yet found, and the push follows the presence loss alone. Starting at sigma 0.9 with 0.02 RMS per step kept the photo and moved the size score from 0.44 to 0.87 (target 0.95). This cell runs that setting, and a stronger one, on the same 10 prompts (Stage 1 only, about 60 minutes per setting; stop after the first if time is short). It writes to `{OUT}/exp3_layout` and makes one sheet per prompt: SD3 | each setting. Send me `exp3_layout.zip`; if a setting changes the images in the right way, 15a is rerun with it for all three rows.

In [ ]:
# 15d. Stage 1 from sigma 0.9, step size limited by its RMS, at two strengths
SETTINGS = {'rms0.02_hi0.9': '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04',
            'rms0.03_hi0.9': '--sigma-hi 0.9 --clip rms --dmax 0.03 --eta0 0.06'}
FIRST = next(iter(SETTINGS))   # SD3 itself is made once, with the first setting
for name, flags in SETTINGS.items():
    arms = 'unguided,stage1' if name == FIRST else 'stage1'
    !python -I code/run_experiment.py --suite size --group dev --n 10 --arms {arms} --out {OUT}/exp3_layout/{name} {flags} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -4
import json
from pathlib import Path
from PIL import Image, ImageDraw
root = Path(f'{OUT}/exp3_layout')
(root / 'sheets').mkdir(exist_ok=True)
columns = [('SD3', root / FIRST / 'unguided')] + [(n, root / n / 'stage1') for n in SETTINGS]
for n, meta in enumerate(sorted((root / FIRST / 'unguided').glob('*/meta.json'))):
    job, prompt = meta.parent.name, json.loads(meta.read_text())['prompt']
    sheet = Image.new('RGB', (532 * len(columns), 560), 'white')
    draw = ImageDraw.Draw(sheet)
    draw.text((8, 8), prompt, fill='black')
    for i, (label, arm) in enumerate(columns):
        image, m = arm / job / 'image.png', arm / job / 'meta.json'
        if image.exists():
            sheet.paste(Image.open(image).convert('RGB').resize((512, 512)), (i * 532, 40))
        f8 = [f for f in json.loads(m.read_text()).get('finals', []) if f['family'] == 'F8'] if m.exists() else []
        draw.text((i * 532 + 8, 24), f"{label}  size score {f8[0]['s']:.2f}" if f8 and f8[0]['s'] is not None else label, fill='black')
        print(f'{prompt[:40]:40s} {label:8s}', [(f['family'], round(f['mu'], 2)) for f in json.loads(m.read_text()).get('finals', [])] if m.exists() else 'missing')
    sheet.save(root / 'sheets' / f'{n:02d}.jpg', quality=88)
!cd {OUT} && zip -qr exp3_layout.zip exp3_layout -x '*.pt'
print('sheets:', root / 'sheets')

### 15e. If 15d is not enough: add the negative prompt (your method only)

The negative prompt is built from the prompt's own constraints (`code/negative.py`): the reversed size relation ("a shoe bigger than a kettle"), "the same size" unless the hedge asks for a small difference, each object alone ("only a kettle", against missing objects), and swapped colours for colour prompts. It goes into the unconditional branch of CFG, only in the arms `stage1_neg` (S1) and `stage1_sar_neg` (FuzzyDiff). Plain SD3 keeps the empty negative prompt.

`unguided_neg` is SD3 with the same negative prompt. Keep it: without it a reviewer cannot tell whether a gain comes from FuzzyDiff or from the negative prompt alone, and the paper must say that only our rows use a negative prompt. Set `LAYOUT_FLAGS` to the 15d setting that worked, or leave it empty. About 1 hour; writes to `{OUT}/exp3_neg`, and a rerun resumes. Don't open `exp3_neg/size.key.json` until you've finished rating.

In [ ]:
# 15e. SD3, SD3 + negative prompt, S1 + negative prompt, FuzzyDiff + negative prompt, then the blind rating page
LAYOUT_FLAGS = ''          # e.g. '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04' if 15d helped
!python -I code/run_experiment.py --suite size --group dev --n 10 --arms unguided,unguided_neg,stage1_neg,stage1_sar_neg --out {OUT}/exp3_neg --resume-from {OUT}/exp3 {LAYOUT_FLAGS} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -12
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import EXP3_NEG_ARMS, build_size_page
build_size_page(f'{OUT}/exp3_neg', Path(f'{OUT}/exp3_neg/size.html'), seed=31, arms=EXP3_NEG_ARMS)
!pip install -q --no-deps piq==0.8.0
!python -I code/auto_metrics.py --run {OUT}/exp3_neg --arms {','.join(EXP3_NEG_ARMS)} --device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|warn' | tail -5
!cd {OUT} && zip -qr exp3_neg.zip exp3_neg -x '*.pt'
print('rating page:', f'{OUT}/exp3_neg/size.html')

In [ ]:
# 15f. Press "Show ratings" on exp3_neg/size.html and paste the text.
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import EXP3_NEG_COMPARISONS, EXP3_NEG_ROWS, exp3_table_rows, score_mixture
human = None
if RATINGS.strip() != 'PASTE HERE':
    Path(f'{OUT}/exp3_neg/size_ratings.json').write_text(RATINGS)
    human = score_mixture(f'{OUT}/exp3_neg/size_ratings.json', f'{OUT}/exp3_neg/size.key.json', field='size',
                          comparisons=EXP3_NEG_COMPARISONS)
    print(json.dumps(human, indent=1))
auto_path = Path(f'{OUT}/exp3_neg/similarity.json')
auto = json.loads(auto_path.read_text()) if auto_path.exists() else None
print(exp3_table_rows(human, auto, EXP3_NEG_ROWS))

## 16. Size violated only (Table 4 column), 20 prompts

The first 20 development size prompts; the first 10 are the Table 4 prompts of section 15. Two questions per image: are both objects there, and does the size relation hold as the prompt words it (including "slightly", "extremely"). A missing object counts as violated.

- **16a** SD3 only (about 15 minutes; SD3 images already in `{OUT}/exp3_v2` are reused). Rate `size20/size_only.html`, then **16b**.
- **16c** later: S1 and FuzzyDiff on the same 20 prompts, at the 15a setting, with a new blind page that mixes all three. Rate that one fresh; don't reuse the SD3-only answers.

Writes only to `{OUT}/size20`. Download `size20.zip`; don't open the `.key.json` files until you've finished rating.

In [ ]:
# 16a. SD3 on 20 size prompts, then a page with only the size questions
!python -I code/run_experiment.py --suite size --group dev --n 20 --arms unguided --out {OUT}/size20 --resume-from {OUT}/exp3_v2 --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -6
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_size_only_page
build_size_only_page(f'{OUT}/size20', Path(f'{OUT}/size20/size_only.html'), seed=41)
!cd {OUT} && zip -qr size20.zip size20 -x '*.pt'
print('rating page:', f'{OUT}/size20/size_only.html')

In [ ]:
# 16b. Press "Show ratings" on size_only.html (16a) or size_all.html (16c) and paste the text.
PAGE = 'size_only'          # 'size_all' after 16c
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import EXP3_COMPARISONS, score_size_only
Path(f'{OUT}/size20/{PAGE}_ratings.json').write_text(RATINGS)
r = score_size_only(f'{OUT}/size20/{PAGE}_ratings.json', f'{OUT}/size20/{PAGE}.key.json', comparisons=EXP3_COMPARISONS)
print(json.dumps({k: r[k] for k in ('size_violation', 'object_missing', 'n_prompts', 'unrated')}, indent=1))
print(json.dumps(r['comparisons'], indent=1))

In [ ]:
# 16c. Later: S1 and FuzzyDiff on the same 20 prompts at the 15a setting, then one blind page with all three
FLAGS = '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04'
!python -I code/run_experiment.py --suite size --group dev --n 20 --arms unguided,stage1,stage1_sar --out {OUT}/size20 --resume-from {OUT}/exp3_v2 {FLAGS} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -12
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import EXP3_ARMS, build_size_only_page
build_size_only_page(f'{OUT}/size20', Path(f'{OUT}/size20/size_all.html'), seed=42, arms=EXP3_ARMS)
!cd {OUT} && zip -qr size20.zip size20 -x '*.pt'
print('rating page:', f'{OUT}/size20/size_all.html')

## 17. Table 2, Exp. (iii), one row at a time

Same prompts as 14d: the 5 size prompts that screening (14a) keeps, where SD3 drew both objects and its size relation fails by FuzzyDiff's own check. Each row is generated into `{OUT}/ablation_size3`, the folder 14d uses, so later rows add to it and 14d resumes instead of regenerating.

- **17a** SD3, no guidance. Runs the size half of 14a (about 30 minutes; SD3 images already in `screen_size` are reused) and always re-screens with `--objects-present`, since an older `screen_size.json` kept prompts with a missing object, then builds `sd3_only.html` with the Exp. (iii) questions: objects present, size relation, Fidelity.
- **17b** paste the ratings; prints the Table 2 rows with the SD3 Exp. (iii) cells filled.

Because the screening keeps prompts SD3 gets wrong, the SD3 row is low by construction; the table must state the screening rule. Rating one row at a time is not blind between rows: the final Table 2 numbers come from the all-arm page of 14d.

In [ ]:
# 17a. Table 2, Exp. (iii): SD3, no guidance, on the screened size prompts
import sys
sys.path.insert(0, 'code')
from pathlib import Path
# SD3 on 30 candidates (resumes: images already in screen_size are kept), then always re-screen: a
# screen_size.json made before --objects-present kept prompts with a missing object
!python -I code/run_experiment.py --suite size --group dev --n 30 --arms unguided --out {OUT}/screen_size --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -6
!python -I code/screen.py --run {OUT}/screen_size --out {OUT}/screen_size.json --keep 5 --objects-present
print(open(f'{OUT}/screen_size.json').read())
!python -I code/run_experiment.py --prompts-file {OUT}/screen_size.json --arms unguided --out {OUT}/ablation_size3 --resume-from {OUT}/screen_size --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -6
from annotate import build_size_page
build_size_page(f'{OUT}/ablation_size3', Path(f'{OUT}/ablation_size3/sd3_only.html'), seed=51, arms=('unguided',))
!cd {OUT} && zip -qr table2_sd3.zip ablation_size3/sd3_only.html ablation_size3/sd3_only.key.json screen_size.json
print('rating page:', f'{OUT}/ablation_size3/sd3_only.html')

In [ ]:
# 17b. Press "Show ratings" on sd3_only.html and paste the text.
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import ablation_table_rows, score_mixture
Path(f'{OUT}/ablation_size3/sd3_only_ratings.json').write_text(RATINGS)
size = score_mixture(f'{OUT}/ablation_size3/sd3_only_ratings.json', f'{OUT}/ablation_size3/sd3_only.key.json',
                     field='size', comparisons=())
print(json.dumps({k: size[k] for k in ('missing', 'size_violation', 'fidelity', 'n_prompts', 'unrated')}, indent=1))
print(ablation_table_rows(size=size))

### 17c. Table 2, Exp. (iii): Stage 1 (full) and its eight ablations, split over two sessions

Same 5 screened prompts as 17a, at the size setting of 14d. About 6 minutes per guided image, so split the nine arms over two Kaggle sessions (about 2 to 2.5 hours each): set `SESSION = 1` in one and `SESSION = 2` in the other. A rerun resumes.

Both sessions must use the same 5 prompts. The session that ran 17a has `screen_size.json`. In another session, either paste its contents (printed by 17a) into `SCREEN`, or leave `SCREEN` as is and the cell screens by itself first (about 30 minutes, same seed and rule as 17a); then check that the printed list matches 17a's.

Every arm gets its own rating page, `ablation_size3/<arm>.html`, with its 5 images. Each session zips its pages as `table2_session<N>.zip`. Rate each page, then paste all ratings into **17d** (in any session that has the key files, or send them to Claude). The two validity-gate rows have no arm and equal Stage 1.

One page per arm is not blind between arms: the rater knows which arm a page shows. Fine for checking; say so if these numbers are reported.

In [ ]:
# 17c. Stage 1 block, one share of the arms per session, one rating page per arm
SESSION = 1                 # 1 in one Kaggle session, 2 in the other
SCREEN = r'''PASTE HERE'''  # optional: the JSON list 17a printed; left as is, this session screens itself
size_flags = '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04'
SHARES = {1: ['stage1', 'abl_point', 'abl_one_sided', 'abl_attention_only'],
          2: ['abl_overlap', 'abl_cond_x0', 'abl_floored', 'abl_weighted_sum', 'abl_no_acceptance']}
import os, sys
sys.path.insert(0, 'code')
from pathlib import Path
if not os.path.exists(f'{OUT}/screen_size.json'):
    Path(OUT).mkdir(parents=True, exist_ok=True)
    if SCREEN.strip() != 'PASTE HERE':
        Path(f'{OUT}/screen_size.json').write_text(SCREEN)
    else:   # screen here, as 17a does (about 30 minutes); compare the printed list with 17a's
        !python -I code/run_experiment.py --suite size --group dev --n 30 --arms unguided --out {OUT}/screen_size --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -6
        !python -I code/screen.py --run {OUT}/screen_size --out {OUT}/screen_size.json --keep 5 --objects-present
print(open(f'{OUT}/screen_size.json').read())
arms = ','.join(SHARES[SESSION])
!python -I code/run_experiment.py --prompts-file {OUT}/screen_size.json --arms {arms} {size_flags} --out {OUT}/ablation_size3 --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -20
from annotate import build_size_page
pages = []
for i, arm in enumerate(SHARES[SESSION]):
    if not list(Path(f'{OUT}/ablation_size3/{arm}').glob('*/image.png')):
        print('no images yet for', arm)
        continue
    build_size_page(f'{OUT}/ablation_size3', Path(f'{OUT}/ablation_size3/{arm}.html'), seed=60 + i, arms=(arm,))
    pages += [f'ablation_size3/{arm}.html', f'ablation_size3/{arm}.key.json']
!cd {OUT} && zip -qr table2_session{SESSION}.zip {' '.join(pages)} screen_size.json
print('rating pages:', [p for p in pages if p.endswith('.html')])

In [ ]:
# 17d. Paste the "Show ratings" text of each arm's page. Leave a page as PASTE HERE until it is rated.
RATINGS = {
    'stage1': r'''PASTE HERE''', 'abl_point': r'''PASTE HERE''', 'abl_one_sided': r'''PASTE HERE''',
    'abl_attention_only': r'''PASTE HERE''', 'abl_overlap': r'''PASTE HERE''', 'abl_cond_x0': r'''PASTE HERE''',
    'abl_floored': r'''PASTE HERE''', 'abl_weighted_sum': r'''PASTE HERE''', 'abl_no_acceptance': r'''PASTE HERE''',
}
import json, os, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import ablation_table_rows, score_mixture
size = {'missing': {'strict': {}, 'lenient': {}}, 'size_violation': {}, 'fidelity': {}}
for arm, text in RATINGS.items():
    key = f'{OUT}/ablation_size3/{arm}.key.json'
    if text.strip() == 'PASTE HERE' or not os.path.exists(key):
        continue
    Path(f'{OUT}/ablation_size3/{arm}_ratings.json').write_text(text)
    r = score_mixture(f'{OUT}/ablation_size3/{arm}_ratings.json', key, field='size', comparisons=())
    for kind in ('strict', 'lenient'):
        size['missing'][kind].update(r['missing'][kind])
    size['size_violation'].update(r['size_violation'])
    size['fidelity'].update(r['fidelity'])
    print(arm, {'missing_strict': r['missing']['strict'].get(arm), 'size_violation': r['size_violation'].get(arm),
                'fidelity': r['fidelity'].get(arm), 'n_prompts': r['n_prompts'], 'unrated': r['unrated']})
print(ablation_table_rows(size=size))

## 18. Standard benchmarks (Table 38): GenEval and T2I-CompBench++

**Start with the pilot (18p):** 1 prompt per benchmark, 1 image, about 40 minutes, to check the whole chain. Then the full runs.

Generation here, on each benchmark's own prompts and image counts; scoring with each benchmark's official code (18d, 18e). Image k of a prompt is seed k, the same for every method.

| Benchmark | Prompts | Images per prompt | Images per method |
|---|---|---|---|
| GenEval | 553 | 4 | 2,212 |
| T2I-CompBench++ Color, Texture, Shape, Spatial | about 300 each | 10 | about 3,000 each |

**Cost.** On 2x T4: SD3 about 40 s per image, FuzzyDiff (Stage 1 + SAR, which also gives the Stage 1 row) about 400 s. GenEval is about 25 GPU hours for SD3 and about 250 for FuzzyDiff; each T2I-CompBench++ category about 33 and 340. Kaggle's quota is about 30 hours a week, so the FuzzyDiff rows need rented or lab GPUs. Every cell resumes; time 5 prompts first and budget from that.

**Scope.** The prompt parser reads colour binding and some spatial words. Texture and shape words are not guided (presence only), and GenEval counting is not guided. Those columns measure FuzzyDiff on prompts it only partly reads; say so with the table. GenEval's "a photo of" frame is removed before parsing (predicates.PHOTO_FRAME); the image is still generated from the full prompt.

**Calibrate first.** These numbers belong to the final model. SD 1.5, Attend-and-Excite and SPO are not run here: take their published scores, with citations and the evaluation version.

In [ ]:
# 18a. Official prompt files, and the prompt lists run_experiment.py reads
!git clone -q --depth 1 https://github.com/djghosh13/geneval.git /kaggle/working/geneval 2>/dev/null || true
!git clone -q --depth 1 https://github.com/Karine-Huang/T2I-CompBench.git /kaggle/working/T2I-CompBench 2>/dev/null || true
GENEVAL_META = '/kaggle/working/geneval/prompts/evaluation_metadata.jsonl'
T2I_DATA = '/kaggle/working/T2I-CompBench/examples/dataset'
!python -I code/benchmarks.py prompts --geneval {GENEVAL_META} --out {OUT}/bench_prompts_geneval.json
for cat in ('color', 'texture', 'shape', 'spatial'):
    !python -I code/benchmarks.py prompts --t2i {T2I_DATA}/{cat}_val.txt --out {OUT}/bench_prompts_{cat}.json


In [ ]:
# 18p. Pilot: 1 random prompt per benchmark, 1 image each, every arm, then export. Checks that generation,
# export and the official scoring (18d, 18e with EXPORT set to the pilot folder) work end to end before any
# GPU money is spent. About 40 minutes on 2x T4. These numbers are not benchmark scores: never report them.
import os
FLAGS = '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04'
for bench in ('geneval', 'color', 'texture', 'shape', 'spatial'):
    source = f'--geneval {GENEVAL_META}' if bench == 'geneval' else f'--t2i {T2I_DATA}/{bench}_val.txt'
    plist = f'{OUT}/bench_pilot_prompts_{bench}.json'
    !python -I code/benchmarks.py prompts {source} --n 1 --out {plist}
    !python -I code/run_experiment.py --prompts-file {plist} --seeds 0 --arms unguided,stage1,stage1_sar --out {OUT}/bench_pilot_{bench} {FLAGS} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -4
    for arm in ('unguided', 'stage1', 'stage1_sar'):
        !python -I code/benchmarks.py export --run {OUT}/bench_pilot_{bench} --arm {arm} {source} --images 1 --only {plist} --out {OUT}/bench_pilot_export/{bench}/{arm}
!cd {OUT} && zip -qr bench_pilot_export.zip bench_pilot_export
!find {OUT}/bench_pilot_export -name '*.png' | sort


In [ ]:
# 18b. Generate one benchmark for one group of arms. Run once per (BENCH, ARMS); each run resumes.
BENCH = 'geneval'            # 'geneval', 'color', 'texture', 'shape' or 'spatial'
ARMS = 'unguided'            # SD3 first (cheap); then 'stage1,stage1_sar' for the two FuzzyDiff rows
LIMIT = ''                   # e.g. '--limit 20' to time a few images first
FLAGS = '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04'
SEEDS = ','.join(str(k) for k in range(4 if BENCH == 'geneval' else 10))
resume = f'--resume-from {RESUME}/bench_{BENCH}' if RESUME else ''
!python -I code/run_experiment.py --prompts-file {OUT}/bench_prompts_{BENCH}.json --seeds {SEEDS} --arms {ARMS} --out {OUT}/bench_{BENCH} {FLAGS} {LIMIT} {resume} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn|^Prompt:|^Objects:|^  |^Outside|^Recognized' | tail -8
!cd {OUT} && zip -qr bench_{BENCH}.zip bench_{BENCH} -x '*.pt'


In [ ]:
import os
# 18c. Export every finished arm into the official layouts: {OUT}/bench_export/<bench>/<arm>/
for bench in ('geneval', 'color', 'texture', 'shape', 'spatial'):
    for arm in ('unguided', 'stage1', 'stage1_sar'):
        if not os.path.isdir(f'{OUT}/bench_{bench}/{arm}'):
            continue
        source = f'--geneval {GENEVAL_META}' if bench == 'geneval' else f'--t2i {T2I_DATA}/{bench}_val.txt'
        !python -I code/benchmarks.py export --run {OUT}/bench_{bench} --arm {arm} {source} --out {OUT}/bench_export/{bench}/{arm}
!cd {OUT} && zip -qr bench_export.zip bench_export


In [ ]:
# 18d. GenEval official scoring. Its pinned environment (mmdet 2.x, Mask2Former) conflicts with this notebook's,
# so run this in a fresh session with bench_export.zip attached as a Dataset; set EXPORT to its folder.
# Commands from the GenEval README; not run by us yet.
EXPORT = f'{OUT}/bench_export'      # f'{OUT}/bench_pilot_export' for the pilot (18p)
!cd /kaggle/working/geneval && conda env create -q -f environment.yml
!cd /kaggle/working && git clone -q https://github.com/open-mmlab/mmdetection.git && cd mmdetection && git checkout -q 2.x && conda run -n geneval pip install -q -v -e .
!cd /kaggle/working/geneval && ./evaluation/download_models.sh /kaggle/working/geneval_detector/
for arm in ('unguided', 'stage1', 'stage1_sar'):
    !cd /kaggle/working/geneval && conda run -n geneval python evaluation/evaluate_images.py {EXPORT}/geneval/{arm} --outfile {OUT}/geneval_{arm}.jsonl --model-path /kaggle/working/geneval_detector/
    !cd /kaggle/working/geneval && conda run -n geneval python evaluation/summary_scores.py {OUT}/geneval_{arm}.jsonl


In [ ]:
# 18e. T2I-CompBench++ official scoring: BLIP-VQA for color, texture and shape; UniDet for spatial.
# Fresh session, its own requirements; see the T2I-CompBench README for the model downloads. Commands from
# that README; not run by us yet. Each scorer reads <dir>/samples and writes its score under <dir>.
EXPORT = f'{OUT}/bench_export'      # f'{OUT}/bench_pilot_export' for the pilot (18p)
for cat in ('color', 'texture', 'shape'):
    for arm in ('unguided', 'stage1', 'stage1_sar'):
        !cd /kaggle/working/T2I-CompBench/BLIPvqa_eval && python BLIP_vqa.py --out_dir={EXPORT}/{cat}/{arm}
        !cat {EXPORT}/{cat}/{arm}/annotation_blip/blip_vqa_score.txt; echo " <- {cat} {arm}"
for arm in ('unguided', 'stage1', 'stage1_sar'):
    !cd /kaggle/working/T2I-CompBench/UniDet_eval && python 2D_spatial_eval.py --outpath {EXPORT}/spatial/{arm}/
    !cat {EXPORT}/spatial/{arm}/labels/annotation_obj_detection_2d/avg_score.txt; echo " <- spatial {arm}"


## 19. Table 13: presence evidence on stress subsets

40 plain SD3 images (10 ordinary, 10 small object, 10 crowded, 10 context scenes such as an empty lake), about 30 minutes. Each image is asked about its named objects, any object the scene suggests, and one random absent object. You label every image-object pair yourself (19b); an object printed on another or merged into it counts as absent. 19c gives AUROC / Brier / ECE per evidence and subset. Global and region evidence are mapped to probabilities by a logistic fit on the other folds (grouped by prompt); the combined row is the model the method uses.


In [ ]:
# 19a. Generate the 40 images, score presence evidence on each, and build the labelling page
!python -I code/presence_eval.py generate --out {OUT}/presence --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -5
!python -I code/presence_eval.py page --out {OUT}/presence
!cd {OUT} && zip -qr presence.zip presence/labels.html presence/presence.jsonl presence/images
print('labelling page:', f'{OUT}/presence/labels.html')


In [ ]:
# 19b. Press "Show labels" on labels.html and paste the text. 19c. Score it into the Table 13 rows.
LABELS = r'''PASTE HERE'''
from pathlib import Path
if LABELS.strip() != 'PASTE HERE':
    Path(f'{OUT}/presence/labels.json').write_text(LABELS)
    !python -I code/presence_eval.py score --out {OUT}/presence --labels {OUT}/presence/labels.json 2>&1 | tail -8


## 19. Table 3, Concept Sliders row: the demo's base model, no slider

No released Concept Slider grades lightness, colour mixture or relative size, and none is trained here, so this row is plain SDXL base with the settings of `demo_concept_sliders.ipynb` (fp16, 50 steps, guidance 5, 1024 px), the full ladder prompt, seed 0. 10 bases per family x 5 hedges = 150 images, split over the two T4s (about 1 h, an estimate). Name the row accordingly in the paper (e.g. "SDXL (Concept Sliders base, no slider)").


In [ ]:
# 19a. Generate the 150 ladder images on both GPUs (a rerun skips finished images)
import subprocess
procs = [subprocess.Popen(f'python -I concept_sliders/generate.py --out {OUT} --device cuda:{i} --part {i} --parts 2 > {OUT}/sdxl_part{i}.log 2>&1', shell=True) for i in (0, 1)]
print([p.wait() for p in procs])
!tail -2 {OUT}/sdxl_part0.log {OUT}/sdxl_part1.log


In [ ]:
# 19b. rho by OWLv2 and one blind rating page for Pair
import sys
sys.path.insert(0, 'code')
from pathlib import Path
RUNS = [f'{OUT}/ladder_attribute', f'{OUT}/ladder_mixture', f'{OUT}/ladder_size_v2']
!python -I code/ladder.py --runs {','.join(RUNS)} --arms sdxl --out {OUT}/ladder_sdxl_auto.json --device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|warn' | tail -50
from ladder import build_ladder_page
build_ladder_page(RUNS, Path(f'{OUT}/ladder_sdxl.html'), seed=14, arms=('sdxl',))
!cd {OUT} && zip -qr ladder_sdxl.zip ladder_sdxl.html ladder_sdxl.key.json ladder_sdxl_auto.json ladder_sdxl_auto.jsonl
print('rating page:', f'{OUT}/ladder_sdxl.html')


In [ ]:
# 19c. Press "Show ratings" on ladder_sdxl.html and paste the text: Pair per family
RATINGS = r'''PASTE HERE'''
import json
from ladder import score_ladder
if RATINGS.strip() != 'PASTE HERE':
    Path(f'{OUT}/ladder_sdxl_ratings.json').write_text(RATINGS)
    print(json.dumps(score_ladder(f'{OUT}/ladder_sdxl_ratings.json', f'{OUT}/ladder_sdxl.key.json'), indent=1))
